# turbopuffer in 10 minutes, with clinical trials

This notebook shows the four things the TrialScope app does with [turbopuffer](https://turbopuffer.com), using
**8 real studies from ClinicalTrials.gov** typed right into the notebook:

1. **Load** documents, letting turbopuffer embed the text for you
2. **Search** three ways: by keyword, by meaning, and both at once
3. **Filter** to trials a specific patient could actually join
4. **Keep each patient's notes separate**, and search a note the moment it's saved

It's completely self-contained: it creates two new namespaces with a random name for this run, loads its own
sample data, and never reads anything else in your account.

You need a turbopuffer API key. In Colab, click the 🔑 **Secrets** icon on the left, add `TURBOPUFFER_API_KEY`,
and turn on **Notebook access**. Then run the cells top to bottom.

*Personal learning project. The patient and notes are made up; trial records are public data and may be out of date.*

In [ ]:
%pip install -q turbopuffer==3.1.0 pandas

## Connect

Each run gets its own pair of namespaces (a namespace is like a table; it's created on the first write), so two
people can run this at the same time without seeing each other's data.

In [2]:
import os
import uuid
import pandas as pd
import turbopuffer

pd.set_option("display.max_colwidth", 80)

try:
    from google.colab import userdata          # running in Colab
    API_KEY = userdata.get("TURBOPUFFER_API_KEY")
except ImportError:
    API_KEY = os.environ.get("TURBOPUFFER_API_KEY") or input("turbopuffer API key: ")

tpuf = turbopuffer.Turbopuffer(api_key=API_KEY, region="aws-us-east-1")

run_id = uuid.uuid4().hex[:6]
trials_ns = tpuf.namespace(f"quickstart-{run_id}-trials")
notes_ns = tpuf.namespace(f"quickstart-{run_id}-patient-dana")


def show(result, *columns):
    """Turn query results into a small table."""
    rows = [{**r.model_dump(), **(r.model_extra or {})} for r in result.rows]
    df = pd.DataFrame(rows).rename(columns={"$dist": "score"})
    return df[[c for c in ("id", "score", *columns) if c in df]]

print("This run's namespaces:", trials_ns.id, "and", notes_ns.id)

This run's namespaces: quickstart-f5dc05-trials and quickstart-f5dc05-patient-dana


## 1. The data: 8 real trials

Five breast cancer trials (most about HER2-positive disease) and three heart trials. Each one has a `text` field
to search (its title and a sentence of its summary) and a few fields we'll filter on later.

In [3]:
trials = [
    {
        "id": "NCT07407920",
        "title": "Ph2 Study for Optimization of Adjunct Systemic Therapy in HER2+ Patients, MolecularPCR Trial",
        "status": "Recruiting", "min_age": 18, "max_age": 120, "sex": "All",
        "text": "Ph2 Study for Optimization of Adjunct Systemic Therapy in HER2+ Patients, MolecularPCR Trial. This phase II trial tests reduced post surgery (adjuvant) therapy for patients with early breast cancer who have confirmed that the disease has responded completely (pathologic complete response)…",
    },
    {
        "id": "NCT04893109",
        "title": "ATEMPT 2.0: Adjuvant T-DM1 vs TH",
        "status": "Recruiting", "min_age": 18, "max_age": 120, "sex": "All",
        "text": "ATEMPT 2.0: Adjuvant T-DM1 vs TH. This research study is studying how well newly diagnosed breast cancer that has tested positive for a protein called HER2 responds using one of two different combination of HER2-directed therapies as…",
    },
    {
        "id": "NCT06876714",
        "title": "ShortStop-HER2: 12 Months vs. 6 Months of HER2-targeted Medications for People With HER2+ Breast Cancer Who Had a Pathologic Complete Response After Chemotherapy Plus Trastuzumab",
        "status": "Recruiting", "min_age": 18, "max_age": 120, "sex": "All",
        "text": "ShortStop-HER2: 12 Months vs. 6 Months of HER2-targeted Medications for People With HER2+ Breast Cancer Who Had a Pathologic Complete Response After Chemotherapy Plus Trastuzumab. This phase III trial compares 6 months of human epidermal growth factor receptor 2 (HER2)-targeted therapy to 12 months of HER2-targeted therapy for the treatment of HER2-positive (+) breast cancer…",
    },
    {
        "id": "NCT05406635",
        "title": "Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients With Breast Cancer",
        "status": "Active not recruiting", "min_age": 18, "max_age": 90, "sex": "Female",
        "text": "Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients With Breast Cancer. Due to a risk of heart failure during HER2 directed therapy in breast cancer, treatment is monitored with imaging of myocardial function, which is resource demanding for both patients and the health…",
    },
    {
        "id": "NCT03418961",
        "title": "S1501 Dual Observational and Randomized Cohort Study of Patients With Metastatic HER-2+ Breast Cancer at Risk of Cardiac Toxicity",
        "status": "Recruiting", "min_age": 18, "max_age": 120, "sex": "All",
        "text": "S1501 Dual Observational and Randomized Cohort Study of Patients With Metastatic HER-2+ Breast Cancer at Risk of Cardiac Toxicity. This trial has two cohorts of patients with human epidermal growth factor receptor (HER)-2-positive breast cancer that has spread to other places in the body. All patients must be receiving…",
    },
    {
        "id": "NCT07465653",
        "title": "A Safety and Tolerability Study of HJB647 in Heart Failure Participants With Reduced Ejection Fraction",
        "status": "Recruiting", "min_age": 18, "max_age": 100, "sex": "All",
        "text": "A Safety and Tolerability Study of HJB647 in Heart Failure Participants With Reduced Ejection Fraction. The purpose of this study is to evaluate the safety, tolerability, and pharmacokinetics of HJB647 at two different doses in participants with chronic stable heart failure with reduced or mildly…",
    },
    {
        "id": "NCT07644806",
        "title": "Olive Oil Supplementation for Heart Failure With Reduced Ejection Fraction",
        "status": "Recruiting", "min_age": 18, "max_age": 85, "sex": "All",
        "text": "Olive Oil Supplementation for Heart Failure With Reduced Ejection Fraction. The OLEA-HF study aims to explore whether daily supplementation of extra-virgin olive oil (EVOO) for 12 weeks during participation in a cardiac rehabilitation program (standard of care) is feasible…",
    },
    {
        "id": "NCT07778966",
        "title": "Community Health Worker Support to Lower Blood Pressure After a Pregnancy With High Blood Pressure",
        "status": "Recruiting", "min_age": 16, "max_age": 56, "sex": "Female",
        "text": "Community Health Worker Support to Lower Blood Pressure After a Pregnancy With High Blood Pressure. High blood pressure during or after pregnancy (such as preeclampsia, gestational hypertension, eclampsia, or Hemolysis, Elevated Liver enzymes, and Low Platelet (HELLP) syndrome) raises a mother's…",
    },
]

pd.DataFrame(trials)[["id", "title", "status", "sex", "min_age", "max_age"]]

,id,title,status,sex,min_age,max_age
0,NCT07407920,"Ph2 Study for Optimization of Adjunct Systemic Therapy in HER2+ Patients, Mo...",Recruiting,All,18,120
1,NCT04893109,ATEMPT 2.0: Adjuvant T-DM1 vs TH,Recruiting,All,18,120
2,NCT06876714,ShortStop-HER2: 12 Months vs. 6 Months of HER2-targeted Medications for Peop...,Recruiting,All,18,120
3,NCT05406635,Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients...,Active not recruiting,Female,18,90
4,NCT03418961,S1501 Dual Observational and Randomized Cohort Study of Patients With Metast...,Recruiting,All,18,120
5,NCT07465653,A Safety and Tolerability Study of HJB647 in Heart Failure Participants With...,Recruiting,All,18,100
6,NCT07644806,Olive Oil Supplementation for Heart Failure With Reduced Ejection Fraction,Recruiting,All,18,85
7,NCT07778966,Community Health Worker Support to Lower Blood Pressure After a Pregnancy Wi...,Recruiting,Female,16,56


## 2. Load them

The one idea to take away: **the `text` field is set up two ways.** `full_text_search` indexes its words for
keyword search, and `embed` tells turbopuffer to turn it into a vector itself. We only ever send plain text.

The other fields are stored as they are and can be filtered on.

In [4]:
result = trials_ns.write(
    upsert_rows=trials,
    distance_metric="cosine_distance",
    schema={"text": {"type": "string", "full_text_search": True, "embed": "voyage/voyage-4-lite"}},
)

print(f"Loaded {result.rows_upserted} trials")

Loaded 8 trials


## 3. Search by keyword (BM25)

Keyword search ranks by the words in the query, giving more weight to rare ones. It's exact: great for gene names,
drug names and trial ids, but it can't match a synonym. Higher score is better.

In [5]:
result = trials_ns.query(
    rank_by=("text", "BM25", "HER2"),
    top_k=3,
    include_attributes=["title"],
)

show(result, "title")

,id,score,title
0,NCT06876714,1.222124,ShortStop-HER2: 12 Months vs. 6 Months of HER2-targeted Medications for Peop...
1,NCT04893109,0.988811,ATEMPT 2.0: Adjuvant T-DM1 vs TH
2,NCT05406635,0.951582,Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients...


## 4. Search by meaning (vector)

Now a question in everyday words: "chemo hurting the heart". The trials never say "chemo" or "hurting".
Vector search compares meaning instead: turbopuffer embeds the question with the same model and returns the
closest trials. Here the score is a distance, so **lower is closer**.

In [6]:
by_meaning = trials_ns.query(
    rank_by=("text", "ANN", ("Embed", "chemo hurting the heart")),
    top_k=3,
    include_attributes=["title"],
)

show(by_meaning, "title")

,id,score,title
0,NCT05406635,0.460516,Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients...
1,NCT03418961,0.525977,S1501 Dual Observational and Randomized Cohort Study of Patients With Metast...
2,NCT07465653,0.570861,A Safety and Tolerability Study of HJB647 in Heart Failure Participants With...


The same question as a keyword search:

In [7]:
by_keyword = trials_ns.query(
    rank_by=("text", "BM25", "chemo hurting the heart"),
    top_k=3,
    include_attributes=["title"],
)

show(by_keyword, "title")

,id,score,title
0,NCT07465653,1.747452,A Safety and Tolerability Study of HJB647 in Heart Failure Participants With...
1,NCT07644806,1.319218,Olive Oil Supplementation for Heart Failure With Reduced Ejection Fraction
2,NCT05406635,1.271325,Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients...


Meaning search puts the two trials about heart problems caused by HER2 cancer treatment first. Keyword
search can only match the word "heart", so it leads with heart failure trials that have nothing to do with
cancer.

## 5. Both at once (hybrid)

The app sends both searches in **one request**, and turbopuffer merges the two ranked lists with reciprocal
rank fusion (`rerank_by=("RRF",)`): each trial scores `1 / (60 + rank)` for every list it appears in, so being
near the top of both lists wins. Higher score is better.

In [8]:
by_both = trials_ns.multi_query(
    queries=[
        {"rank_by": ("text", "ANN", ("Embed", "HER2 breast cancer treatment after surgery")), "top_k": 5, "include_attributes": ["title"]},
        {"rank_by": ("text", "BM25", "HER2 breast cancer treatment after surgery"), "top_k": 5, "include_attributes": ["title"]},
    ],
    rerank_by=("RRF",),
)

show(by_both.results[0], "title")

,id,score,title
0,NCT06876714,0.032522,ShortStop-HER2: 12 Months vs. 6 Months of HER2-targeted Medications for Peop...
1,NCT07407920,0.032522,"Ph2 Study for Optimization of Adjunct Systemic Therapy in HER2+ Patients, Mo..."
2,NCT04893109,0.031498,ATEMPT 2.0: Adjuvant T-DM1 vs TH
3,NCT05406635,0.031258,Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients...
4,NCT03418961,0.015625,S1501 Dual Observational and Randomized Cohort Study of Patients With Metast...
5,NCT07778966,0.015385,Community Health Worker Support to Lower Blood Pressure After a Pregnancy Wi...


## 6. Filter to trials a patient could join

Meet Dana, a made-up patient: 58, female, HER2-positive breast cancer. Filters are exact and run *inside* the
search, so trials she can't join never compete for a spot in the results.

In [9]:
could_join = ("And", [
    ("status", "Eq", "Recruiting"),
    ("min_age", "Lte", 58),
    ("max_age", "Gte", 58),
    ("sex", "In", ["Female", "All"]),
])

for_dana = trials_ns.query(
    rank_by=("text", "ANN", ("Embed", "HER2 breast cancer after surgery")),
    filters=could_join,
    top_k=4,
    include_attributes=["title"],
)

show(for_dana, "title")

,id,score,title
0,NCT07407920,0.371704,"Ph2 Study for Optimization of Adjunct Systemic Therapy in HER2+ Patients, Mo..."
1,NCT06876714,0.463210,ShortStop-HER2: 12 Months vs. 6 Months of HER2-targeted Medications for Peop...
2,NCT04893109,0.511486,ATEMPT 2.0: Adjuvant T-DM1 vs TH
3,NCT03418961,0.525373,S1501 Dual Observational and Randomized Cohort Study of Patients With Metast...


And the trials that filter ruled out, by asking for the opposite with `Not`:

In [10]:
ruled_out = trials_ns.query(
    filters=("Not", could_join),
    top_k=10,
    include_attributes=["status", "min_age", "max_age", "title"],
)

show(ruled_out, "status", "min_age", "max_age", "title")

,id,status,min_age,max_age,title
0,NCT05406635,Active not recruiting,18,90,Imaging Versus Cardiac Biomarker Monitored HER2 Directed Therapy in Patients...
1,NCT07778966,Recruiting,16,56,Community Health Worker Support to Lower Blood Pressure After a Pregnancy Wi...


One isn't recruiting any more, and one only takes patients up to 56. However well their text matches, they
can never show up in Dana's results.

## 7. Each patient gets their own namespace

In the app, every patient's notes live in a separate namespace, so a search of Dana's notes can only ever see
Dana's notes. Here we save four short notes and search them straight away: turbopuffer queries are **strongly
consistent**, so a note is searchable as soon as the write returns.

In [11]:
notes_ns.write(
    upsert_rows=[
        {"id": "n1", "date": "2026-08-12", "text": "Initial consult. New HER2-positive breast cancer, stage II. Plan chemotherapy plus trastuzumab before surgery."},
        {"id": "n2", "date": "2026-08-24", "text": "Baseline echocardiogram. Heart pumping function (LVEF) 64%, normal."},
        {"id": "n3", "date": "2026-10-31", "text": "Echo before cycle 4. LVEF 58%, a small drop from baseline. No symptoms. Keep monitoring every 3 months."},
        {"id": "n4", "date": "2026-11-04", "text": "Follow-up after cycle 4. Mild fatigue. Asked about clinical trials after surgery."},
    ],
    distance_metric="cosine_distance",
    schema={"text": {"type": "string", "full_text_search": True, "embed": "voyage/voyage-4-lite"}},
)

result = notes_ns.query(
    rank_by=("text", "ANN", ("Embed", "is her heart okay?")),
    top_k=2,
    include_attributes=["date", "text"],
)

show(result, "date", "text")

,id,score,date,text
0,n2,0.519428,2026-08-24,"Baseline echocardiogram. Heart pumping function (LVEF) 64%, normal."
1,n3,0.594263,2026-10-31,"Echo before cycle 4. LVEF 58%, a small drop from baseline. No symptoms. Keep..."


## How this maps to the app

| In this notebook | In TrialScope |
| --- | --- |
| `text` field with `full_text_search` and `embed` | Every corpus: trials, PubMed abstracts, notes |
| Keyword, vector and hybrid search | The search page's mode switch (hybrid is the default) |
| Reciprocal rank fusion | How hybrid lists, and results from several sources, are combined |
| Filters from the patient's age and sex | "Match patient" on trial search, and the copilot's trial matching |
| One namespace per patient | `ts-patient-0001` … `ts-patient-0100` |

The copilot uses exactly these queries as its tools: it decides what to search, with which filters, and cites
what comes back.

## Clean up

Deletes this run's two namespaces. Nothing else in the account is touched.

In [12]:
trials_ns.delete_all()
notes_ns.delete_all()

print("Deleted", trials_ns.id, "and", notes_ns.id)

Deleted quickstart-f5dc05-trials and quickstart-f5dc05-patient-dana
